## Feature engineering

In [63]:
import pandas as pd
import numpy as np
from config import ORIG

In [64]:
print(ORIG)

C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data


### 1. Import data

In [ ]:
df_cal = pd.read_csv(ORIG / "calibration_transactions.csv")
pd.set_option('display.max_columns', None)

In [51]:
df_cal.info()
df_cal.shape #(220515, 23)

<class 'pandas.DataFrame'>
RangeIndex: 220515 entries, 0 to 220514
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   cust_id                220515 non-null  str    
 1   order_date             220515 non-null  str    
 2   pack_date              220515 non-null  str    
 3   sale_id                220515 non-null  str    
 4   sale_discount_applied  220515 non-null  float64
 5   sale_revenue           220515 non-null  float64
 6   returned_to_shop_id    41134 non-null   str    
 7   prod_id                220515 non-null  str    
 8   prod_size              220515 non-null  float64
 9   prod_web_only          220515 non-null  int64  
 10  prod_season            220515 non-null  str    
 11  prod_brand             220515 non-null  str    
 12  prod_title             220515 non-null  str    
 13  prod_color             220515 non-null  str    
 14  prod_type_1            220515 non-null  str    

(220515, 23)

### 2. Quick inspect of the variables for the feartures 
(some additional data cleaning steps, may need to be added to the data cleaning file)

In [52]:
df_cal["cust_id"].nunique() # 93272 unique customers
df_cal["cust_id"].count() # 220515 transactions, so on average ~2.37 transactions per customer
df_cal["prod_web_only"].value_counts() #170115 are not web only, 50400 are web only

prod_web_only
0    170115
1     50400
Name: count, dtype: int64

In [53]:
df_cal["prod_material"].value_counts() # a lot of options > make a new material variabel with only the first category 
df_cal["material_main"] = df_cal["prod_material"].str.split(",").str[0].str.strip()
df_cal["material_main"].value_counts() 


material_main
leather              92221
textile              39046
synthetic leather    31494
suede                20831
nubuck                9186
patent leather        5977
synthetic             4507
rubber                1311
wool                   716
velour                 301
synthetic wool         280
Name: count, dtype: int64

In [54]:
df_cal["prod_type_1"].value_counts() # 4 categories (Women, Men, Girls and Boys)
df_cal["prod_type_3"].value_counts() # a lot of categories, only take the first category 
df_cal["prod_type_3_main"] = df_cal["prod_type_3"].str.split(",").str[0].str.strip()
df_cal["prod_type_3_main"].value_counts() 
df_cal["prod_type_5"].value_counts()

prod_type_5
low-top sneakers          30310
lace-up shoes             15295
lace-up boots             14263
dress sandals             13538
ankle boots               11799
slippers                   9108
classic pumps              9011
high-top sneakers          9009
sport sandals              7723
casual shoes               6744
slides                     6378
boots                      6116
slip-on shoes              5691
flip-flops                 5611
baby shoes                 5546
dress boots                4932
running sneakers           4695
chunky boots               4384
classic ballet flats       4124
wedge sandals              4005
high-heel pumps            3710
velcro shoes               3544
slip-on sneakers           2872
boots (velcro)             2690
strap pumps                2505
strap ballet flats         2413
rain boots                 1972
boat shoes                 1693
platform pumps             1494
wedge sneakers             1457
sports shoes               1

In [ ]:
# SEASON
df_cal["prod_season"].value_counts() # a lot of categories > make new variabels ( (1) seasons: winter, summer, NOS > never out of stock and CONS > basics; (2) years and (3) type of collection)

# extract season code 
df_cal["season_code"] = df_cal["prod_season"].str.extract(r'^([A-Z]+)')

# extract year (last 2 digits if present) and convert to full year
df_cal["season_year"] = df_cal["prod_season"].str.extract(r'(\d{2})$')
df_cal["season_year"] = df_cal["season_year"].astype(float) + 2000

# mapping to season names
season_map = {
    "Z": "Summer",
    "W": "Winter",
    "MZ": "Mid-season Summer",
    "MW": "Mid-season Winter",
    "SZ": "Special Summer",
    "SW": "Special Winter",
    "NOS": "Never Out of Stock",
    "CONS": "Consignment"
}

df_cal["season"] = df_cal["season_code"].map(season_map)

# mapping to type (main season, mid-season, ... )
type_map = {
    "Z": "Main",
    "W": "Main",
    "MZ": "Mid-season",
    "MW": "Mid-season",
    "SZ": "Special",
    "SW": "Special",
    "NOS": "NOS",
    "CONS": "CONS"
}

df_cal["season_type"] = df_cal["season_code"].map(type_map)

In [60]:
df_cal.head()

,cust_id,order_date,pack_date,sale_id,sale_discount_applied,sale_revenue,returned_to_shop_id,prod_id,prod_size,prod_web_only,prod_season,prod_brand,prod_title,prod_color,prod_type_1,prod_type_3,prod_type_5,prod_material,prod_outlet,prod_size_bin,discount_quantile,revenue_quantile,revenue_2018_2019,material_main,prod_type_3_main,season_code,season_year,season,season_type
0,klantwj2374mzmab,2016-01-01,2016-01-04,wa46phiwo6neterg,-35.70,83.30,NaN,543rj4mzzjnzkbil,30.0,0,W14,STONES and BONES,Bruine Velcrobottine STONES and BONES,dark brown,boys,high shoes,boots (velcro),leather,0,1,Q2 (20-40%),Q5 (80-100%),209.85,leather,high shoes,W,2014.0,Winter,Main
1,klantwj2374mzmab,2016-01-01,2016-01-04,wa46phiwo6neterg,-39.98,0.00,ztodvuuaje,7ewnqhtrquent4cy,44.0,0,W15,Mario Rossi,Mario Rossi Cognac Boot,cognac,men,high shoes,dress boots,leather,0,2,Q1 (0-20%),Q1 (0-20%),209.85,leather,high shoes,W,2015.0,Winter,Main
2,dt7cthjqnjmkbiu6,2016-01-01,2016-01-05,lneitdgyfvrie3jo,-29.98,0.00,geja5b25na,5ut47kvr6zlx6y62,25.0,0,W15,STONES and BONES,STONES and BONES Taupe Velcroboots,taupe,boys,high shoes,boots (velcro),NaN,0,1,Q2 (20-40%),Q1 (0-20%),0.00,NaN,high shoes,W,2015.0,Winter,Main
3,dt7cthjqnjmkbiu6,2016-01-01,2016-01-05,lneitdgyfvrie3jo,-29.98,69.97,NaN,43q23l7qojmq4phw,25.0,0,W14,CKS,Blauwe CKS Bottines,dark blue,boys,high shoes,lace-up boots,NaN,0,1,Q2 (20-40%),Q4 (60-80%),0.00,NaN,high shoes,W,2014.0,Winter,Main
4,etcrrgcbrzfovyzj,2016-01-01,2016-01-08,qma4mxuh7xopy4um,-82.50,82.50,NaN,yucgdvgqaan7672w,37.0,0,W13,Novi Due,Zwarte Novi Due Lange Laars,black,women,boots,boots,NaN,0,2,Q1 (0-20%),Q5 (80-100%),0.00,NaN,boots,W,2013.0,Winter,Main


Every customer has possibly multiple rows (one per item bought) and multiple orders over time. 

Sales_revenue is often 0, this is the case when the item has been returned. But the discount of that item is still in the df, so will impact the features (especially the revenue or total discount). So return items are not used when calculating the revenue, ...

### 3. Features pipeline


In [61]:
# Feature pipeline
def build_customer_features(df):
    df = df.copy()

    # Dates and returns
    df["order_date"] = pd.to_datetime(df["order_date"])
    if "is_returned" not in df.columns:
        if "returned_to_shop_id" in df.columns:
            df["is_returned"] = df["returned_to_shop_id"].notna().astype(int)
        else:
            df["is_returned"] = 0

    cutoff_date = df["order_date"].max() + pd.Timedelta(days=1)

    # Only non-returned items for discount and revenue featues
    df_non_returned = df[df["is_returned"] == 0]

    # Basic features
    agg = df.groupby("cust_id").agg(
        n_items=("sale_id", "count"),
        n_sales=("sale_id", "nunique"),
        n_returns=("is_returned", "sum"),
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max"),
    )

    # Returned items/sales
    agg["n_items_non_returned"] = df_non_returned.groupby("cust_id")["sale_id"].count()
    agg["n_sales_non_returned"] = df_non_returned.groupby("cust_id")["sale_id"].nunique()

    # Revenue (net, gross and discount)
    agg["total_revenue_net"] = df_non_returned.groupby("cust_id")["sale_revenue"].sum()
    agg["total_discount"] = df_non_returned.groupby("cust_id")["sale_discount_applied"].apply(lambda s: s.abs().sum())
    agg["total_revenue_gross"] = agg["total_revenue_net"] + agg["total_discount"]

    # Derived features
    agg["customer_lifetime_days"] = (agg["last_purchase"] - agg["first_purchase"]).dt.days
    agg["recency_days"] = (cutoff_date - agg["last_purchase"]).dt.days
    agg["avg_rev_per_item"] = agg["total_revenue_net"] / agg["n_items_non_returned"]
    agg["avg_rev_per_sale"] = agg["total_revenue_net"] / agg["n_sales_non_returned"]
    agg["return_rate"] = agg["n_returns"] / agg["n_items"]
    agg["discount_ratio"] = agg["total_discount"] / (agg["total_revenue_net"] + agg["total_discount"])

    # Brands 
    brand_counts = df.groupby(["cust_id", "prod_brand"]).size().unstack(fill_value=0)
    brand_share = brand_counts.div(brand_counts.sum(axis=1), axis=0)
    agg["top_brand_share"] = brand_share.max(axis=1)

    # Web-only
    agg["share_web_only"] = df.groupby("cust_id")["prod_web_only"].mean()

    # Shoesize
    size_stats = df.groupby("cust_id")["prod_size"].agg(
        size_mean="mean",
        size_std=lambda s: s.std(ddof=0),  # ddof=0 -> std=0 bij 1 observatie
        size_min="min",
        size_max="max",
    )
    agg = agg.join(size_stats)
    agg["size_std"] = agg["size_std"].fillna(0)

    # Season (Winter/Summer + NOS/CONS)
    if "season" in df.columns:
        s = df["season"].astype(str).str.strip().str.lower()
        is_summer = s.eq("summer")
        is_winter = s.eq("winter")
        is_other = s.isin(["nos", "cons"])  # NOS / CONS

        agg["share_summer_items"] = df.assign(_is_summer=is_summer).groupby("cust_id")["_is_summer"].mean()
        agg["share_winter_items"] = df.assign(_is_winter=is_winter).groupby("cust_id")["_is_winter"].mean()
        agg["share_other_season_items"] = df.assign(_is_other=is_other).groupby("cust_id")["_is_other"].mean()

    # Season type shares (main, mid-season, special, other) 
    if "season_type" in df.columns:
        st = df["season_type"].astype(str).str.strip().str.lower()
        agg["share_season_type_main"] = df.assign(_st_main=st.eq("main")).groupby("cust_id")["_st_main"].mean()
        agg["share_season_type_mid_season"] = df.assign(_st_mid=st.eq("mid-season")).groupby("cust_id")["_st_mid"].mean()
        agg["share_season_type_special"] = df.assign(_st_special=st.eq("special")).groupby("cust_id")["_st_special"].mean()
        agg["share_season_type_other"] = df.assign(_st_other=st.eq("other")).groupby("cust_id")["_st_other"].mean()

    # Luxery proxy (price quantile per item) ---
    if "revenue_quantile" in df.columns:
        high_map = {"Q5 (80-100%)": 1, "Q4 (60-80%)": 1}
        df["is_high_revenue_item"] = df["revenue_quantile"].map(high_map).fillna(0).astype(int)
        agg["share_high_revenue_items"] = df.groupby("cust_id")["is_high_revenue_item"].mean()

    # Material
    if "material_main" in df.columns:
        mat_main_counts = df.groupby(["cust_id", "material_main"]).size().unstack(fill_value=0)
        mat_main_share = mat_main_counts.div(mat_main_counts.sum(axis=1), axis=0)
        agg["material_main_dominant_share"] = mat_main_share.max(axis=1)
        agg["material_main_n_unique"] = (mat_main_counts > 0).sum(axis=1)

    return agg.reset_index()


In [62]:
# Implement features
features = build_customer_features(df_cal)
features.head()

,cust_id,n_items,n_sales,n_returns,first_purchase,last_purchase,n_items_non_returned,n_sales_non_returned,total_revenue_net,total_discount,total_revenue_gross,customer_lifetime_days,recency_days,avg_rev_per_item,avg_rev_per_sale,return_rate,discount_ratio,top_brand_share,share_web_only,size_mean,size_std,size_min,size_max,share_summer_items,share_winter_items,share_other_season_items,share_season_type_main,share_season_type_mid_season,share_season_type_special,share_season_type_other,share_high_revenue_items,material_main_dominant_share,material_main_n_unique
0,222agnowc53dykbq,1,1,0,2016-12-13,2016-12-13,1.0,1.0,89.95,0.00,89.95,0,384,89.950,89.95,0.000000,0.000000,1.000000,0.0,46.0,0.000000,46.0,46.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.000000,1.000000,1.0
1,222ny4m63rmalpdl,3,1,1,2016-12-22,2016-12-22,2.0,1.0,125.93,33.97,159.90,0,375,62.965,125.93,0.333333,0.212445,0.666667,0.0,38.0,3.559026,33.0,41.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.666667,0.666667,2.0
2,223jend5smd4ptmc,1,1,0,2016-08-30,2016-08-30,1.0,1.0,89.95,0.00,89.95,0,489,89.950,89.95,0.000000,0.000000,1.000000,0.0,37.0,0.000000,37.0,37.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.000000,1.000000,1.0
3,223xvc4rbjatlnev,2,1,0,2016-07-29,2016-07-29,2.0,1.0,116.14,49.76,165.90,0,521,58.070,116.14,0.000000,0.299940,1.000000,0.0,37.5,0.500000,37.0,38.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.500000,1.000000,1.0
4,223y2r357elerbis,2,1,1,2017-01-17,2017-01-17,1.0,1.0,47.97,31.98,79.95,0,349,47.970,47.97,0.500000,0.400000,1.000000,0.0,40.0,0.000000,40.0,40.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.000000,1.000000,1.0


In [59]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 93272 entries, 0 to 93271
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   cust_id                       93272 non-null  str           
 1   n_items                       93272 non-null  int64         
 2   n_sales                       93272 non-null  int64         
 3   n_returns                     93272 non-null  int64         
 4   first_purchase                93272 non-null  datetime64[us]
 5   last_purchase                 93272 non-null  datetime64[us]
 6   n_items_non_returned          93267 non-null  float64       
 7   n_sales_non_returned          93267 non-null  float64       
 8   total_revenue_net             93267 non-null  float64       
 9   total_discount                93267 non-null  float64       
 10  total_revenue_gross           93267 non-null  float64       
 11  customer_lifetime_days        93272 non

In [ ]:
#run to get csv file with features
features.to_csv("data/features.csv", index=False)

## Information freatures 
### 1. Basic Features 
**n_item**: amount of bought items 

**n_sales**: amount of unique sales

**n_returns**: amount of returned items (sum of is_returned (=1 if returned and empty if not returned))

**first_purchase**: date of first purchase

**last_purchase**: date of last purchase 

**n_items_non_returned**: amount of items that are not returned 

**n_sales_non_returned**: amount of sales not returned

**total_revenue_net**: revenue for each customer (after the discount) (returned items not included)

**total_discount**: total disount over all orders of a customer (returned items not included)

**total_revenue_gross**: price prior to discount (returned items not included)


### 2. Derived Features 
**customer_lifetime_days**: amount of days a customer is active (difference first and last purchase, is 0 when only 0 order) 

**recency_days**: how recent last purchase was

**avg_rev_per_item**: average revenue per item (returned items not included)

**avg_rev_per_sale**: average revenue per sale (returned items not included)

**return_rate**: retour percentage (n_retuns/n_items) 

**discount_ratio**: % of discount compared to full price (before discount) (returned items not included)


### 3. Brands
**top_brand_share**: % of how much one brand is prominent (ex: 1 = all items are of the same brand, ...)


### 4. Web-only 
**share_web_only**: amount of sales web-only (avg of prob_web_only)


### 5. Shoe sizes 
**size_mean**: average shoe size

**size_std**: std shoe size

**size_min**: smallest shoe size

**size_max**: largest shoe size


### 6. Season
**share_summer_items**: % of items that are of the Summer seaons

**share_winter_items**: % of items that are of the Winter seaons

**share_other_season_items**: % of items that are of NOS and COS

**share_season_type_main**: % of items that are of the main seaons

**share_season_type_mid_season**: % of items that are of the mid-season seaons

**share_season_type_special**: % of items that are of the special season 

**share_season_type_other**: % of items that are of the other season (NOS, COS)


### 7. Luxery proxcy 
**share_high_revenue_items**: share of items in high price quantiles (1 if all items are for revenue_quantile in Q4 or Q5)   


### 8. Materials
**material_main_dominant_share**: % of items from the main material 

**material_main_n_unique**: amount of different materials for a customer